K - fold

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="MedHouseVal")

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    shuffle = True,
    test_size=0.2,
    random_state=42
)

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [6]:
model_standard = LinearRegression()
model_standard.fit(X_train_scaled, y_train)

y_pred_standard = model_standard.predict(X_train_scaled)
mse_standard = mean_squared_error(y_train, y_pred_standard)
r2_standard = r2_score(y_train, y_pred_standard)

In [8]:
# stampiamo delle variabili che ci serviranno in seguito nel plotting

feature_to_plot = "MedInc"
feature_index = X.columns.get_loc(feature_to_plot)
coef_standard = model_standard.coef_[feature_index]
intercept_standard = model_standard.intercept_

In [9]:
K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=42)

In [14]:
# iterazione manuale del k-fold

all_mse_kfold = []
all_r2_kfold = []
kfold_models_coef =[]
last_model_kfold = LinearRegression()

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
    X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]

    fold_scaler = StandardScaler()
    X_train_scaled_fold = fold_scaler.fit_transform(X_train_fold)
    X_test_scaled_fold = fold_scaler.transform(X_test_fold)

    model_fold = LinearRegression()
    model_fold.fit(X_train_scaled_fold, y_train_fold)
    y_pred_fold = model_fold.predict(X_test_scaled_fold)

    all_mse_kfold.append(mean_squared_error(y_test_fold, y_pred_fold))
    all_r2_kfold.append(r2_score(y_test_fold, y_pred_fold))

    kfold_models_coef.append(model_fold.coef_)
    last_model_kfold = model_fold

mean_mse_kfold = np.mean(all_mse_kfold)
mean_r2_kfold = np.mean(all_r2_kfold)
avg_coeff_kfold = np.mean([c[feature_index] for c in kfold_models_coef])
avg_intercept_kfold = last_model_kfold.intercept_

In [15]:
print(f"MSE TEST:\t{mse_standard:.4f}")
print(f"R2 TEST:\t{r2_standard:.4f}")
print(f"MSE k-Fold:\t{mean_mse_kfold:.4f}")
print(f"R2 k-fold:\t{mean_r2_kfold:.4f}")

MSE TEST:	0.5179
R2 TEST:	0.6126
MSE k-Fold:	0.5306
R2 k-fold:	0.6014
